In [ ]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.path.abspath("./"))

print(f"Current work directory: {os.getcwd()}")

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scripts.plotting as pl
import anndata

In [ ]:
adata = sc.read_h5ad('./Data/h5ad/adata_oc43_removed_normalized.h5ad')

In [ ]:
# Compute HVG directly on the entire dataset
sc.pp.highly_variable_genes(adata, flavor='seurat', subset=False, inplace=True)

In [ ]:
adata_hvg = adata[:, adata.var['highly_variable']].copy()

In [ ]:
adata_hvg

In [ ]:
adata_hvg.var

In [ ]:
np.random.seed(42)

sc.tl.pca(adata_hvg, svd_solver='arpack')
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, resolution=0.1)

In [ ]:
pct = adata_hvg.obs['pct_counts_oc43'].astype(float)
tot = adata_hvg.obs['total_counts_oc43'].astype(float)

# Threshold (adjust if necessary)
p_hi = 10.0 # High infection threshold (%)
eps = 0.1 # No infetion threshold (%)
t_hi = 10 # Upper limit of total_counts for No threshold

# Conditional Mask
no_mask   = ((pct <= eps) | (tot <= t_hi)).fillna(False)
high_mask = ((pct >= p_hi) & (tot > t_hi)).fillna(False)

labels = np.select(
    [no_mask, high_mask],
    ['No infection', 'High infection'],
    default='Low infection'
)

adata_hvg.obs['infection_group'] = pd.Categorical(
    labels,
    categories=['No infection','Low infection','High infection'],
    ordered=True
)

adata.obs['infection_group'] = adata_hvg.obs['infection_group']
adata.obs['infection_group'] = adata_hvg.obs['infection_group']
print(adata_hvg.obs['infection_group'].value_counts())

In [ ]:
adata_hvg.obs['condition'] = adata_hvg.obs['condition'].str.replace('_MOI1.0', '').astype('category')

In [ ]:
adata_hvg.obs['condition']

In [ ]:
category_colors = {
        'No infection': '#4A90E2',
        'Low infection': '#F5A623',
        'High infection': '#D0021B'
    }
condition_palette = {
    '00h_uninfected': '#1f77b4',  # blue
    '16h':     '#ff7f0e',  # orange
    '24h':     '#2ca02c',  # green
    '48h':     '#d62728'   # red
}

In [ ]:
## Seed 42
pl.set_publication_style()
pl.plot_umap_categorical(adata_hvg, color_var='infection_group', seed = 42, title= "Infection status",palette=category_colors, save_path='./Plot/Infection_group')
pl.plot_umap_categorical(adata_hvg, color_var='condition', seed = 42, title= "Infection Time course",palette=condition_palette, save_path='./Plot/Infection_teimcourse')

In [ ]:
pl.plot_umap_continuous(adata_hvg, color_var='pct_counts_oc43', title = "Viral Load(% OC43)", save_path='./Plot/Fig1_Viral_Load')

In [ ]:
genes_of_interest = [
   'TPI1','SNHG7','FTL','RPS29','PPIA','IFRD1','HMGN2','TSC22D3','HSPA8','ATP5MF']

In [ ]:
pl.plot_gene_expression_series(adata, 
    genes=genes_of_interest,
    group_key='infection_group',figsize=(8, 12), save_path= "./Plot/Gene_expression_levels_by_infection_status")

In [ ]:
adata.obs['infection_group'] = adata.obs['infection_group'].astype('category')
adata.obs['infection_group'] = adata.obs['infection_group'].cat.reorder_categories(['No infection', 'Low infection', 'High infection'])

dp = sc.pl.dotplot(adata, var_names=genes_of_interest, groupby='infection_group', 
              standard_scale='var', 
              cmap='seismic', 
              categories_order=['No infection', 'Low infection', 'High infection'],
              show=False) 

ax = dp['mainplot_ax']
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.savefig('./Plot/_Fig2D_DotPlot.pdf', dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.savefig('./Plot/_Fig2D_DotPlot.png', dpi=300, bbox_inches='tight', pad_inches=0.2)

print("DotPlot saved")

In [ ]:
violin_palette = {
        'No infection': '#4A90E2', 
        'Low infection': '#F5A623', 
        'High infection': '#D0021B'
    }
target_genes = ['TPI1', 'RPS29', 'PPIA', 'SNHG7', 'TSC22D3', 'IFRD1']
pl.plot_gene_violin_overlay(adata, target_genes, group_key='infection_group', 
                             order=['No infection', 'Low infection', 'High infection'], palette=violin_palette, figsize=(14,8), 
                             save_path='./Plot/_Fig2C_ViolinPlot.png')

In [ ]:
anndata.settings.allow_write_nullable_strings = True
adata_hvg.write_h5ad('./Data/h5ad/Visualize.h5ad')

In [ ]:
import session_info

session_info.show()